In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functools import reduce
from transformers import (
    AutoModel,
    BertTokenizerFast,
    BertConfig,
    BertModel,
    Mamba2Config,
    Mamba2Model,
)
from models import BERT_Arch, BERT_Arch_Fourier, Mamba_Fourier
from gprofiler import GProfiler
from data_utils import clean_feature_column, convert_id, combine_dfs, map_metab
import torch.nn.functional as F

# MPS
if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
num_bins = 40

# get bert
bert_config = BertConfig()
bert = BertModel(bert_config)

# Load the BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [ ]:
with open("data/comb_matrix.npy", "rb") as f:
    matrix_data = np.load(f)

In [ ]:
## TODO: Testing with both datasets combined together
_m = np.nan_to_num(matrix_data, nan=0.0)
matrix_data_conv = torch.log2(
    torch.abs(torch.fft.rfft(torch.FloatTensor(_m), dim=1))[:, :512] + 1
)  ## Using log scaler to check improvement in perf.
# split train dataset into train, validation and test sets
train_text, temp_text, train_labels, temp_labels = train_test_split(
    matrix_data_conv,
    [1] * matrix_data.shape[0],
    random_state=42,  # ;)
    test_size=0.3,
)


val_text, test_text, val_labels, test_labels = train_test_split(
    temp_text, temp_labels, random_state=42, test_size=0.5, stratify=temp_labels
)

tokens_train = train_text
tokens_val = val_text
tokens_test = test_text


max_len = 25

In [ ]:
train_seq = torch.tensor(tokens_train)
train_mask = torch.ones_like(train_seq)
train_y = torch.tensor(train_labels)

val_seq = torch.tensor(tokens_val)
val_mask = torch.ones_like(val_seq)
val_y = torch.tensor(val_labels)

test_seq = torch.tensor(tokens_test)
test_mask = torch.ones_like(test_seq)
test_y = torch.tensor(test_labels)

/var/folders/wn/f0f6wxmj2r7_j0vg75qyxp8h0000gn/T/ipykernel_56783/3036122672.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_seq = torch.tensor(tokens_train)
/var/folders/wn/f0f6wxmj2r7_j0vg75qyxp8h0000gn/T/ipykernel_56783/3036122672.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  val_seq = torch.tensor(tokens_val)
/var/folders/wn/f0f6wxmj2r7_j0vg75qyxp8h0000gn/T/ipykernel_56783/3036122672.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_seq = torch.tensor(tokens_test)


In [ ]:
def apply_random_mask(batch_seq, mask_prob=0.15):
    """
    batch_seq: (batch_size, seq_len)
    Returns: masked_seq, labels, masked_indices
    """
    labels = batch_seq.clone()
    masked_seq = batch_seq.clone()

    probability_matrix = torch.full(masked_seq.shape, mask_prob)
    masked_indices = torch.bernoulli(probability_matrix).bool()

    masked_seq[masked_indices] = 0.0

    return masked_seq, labels, masked_indices

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# define a batch size
batch_size = 1

# wrap tensors
train_data = TensorDataset(train_seq, train_mask, train_y)

# sampler for sampling the data during training
train_sampler = RandomSampler(train_data)

# dataLoader for train set
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

# wrap tensors
val_data = TensorDataset(val_seq, val_mask, val_y)

# sampler for sampling the data during training
val_sampler = SequentialSampler(val_data)

# dataLoader for validation set
val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)

In [ ]:
# (don't) freeze all the parameters
for param in bert.parameters():
    param.requires_grad = True

In [ ]:
# pass the pre-trained BERT to our define architecture
# model = BERT_Arch(bert)
model = BERT_Arch_Fourier(bert)
# push the model to GPU
model = model.to(device)

In [ ]:
# optimizer from hugging face transformers
from torch.optim import AdamW

# define the optimizer
optimizer = AdamW(model.parameters(), lr=1e-5)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# compute the class weights
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_labels), y=train_labels
)
print("Class Weights:", class_weights)

Class Weights: [1.]


In [ ]:
# converting list of class weights to a tensor
weights = torch.tensor(class_weights, dtype=torch.float)

# push to GPU
weights = weights.to(device)

# define the loss function
cross_entropy = nn.NLLLoss(weight=weights)

# number of training epochs
epochs = 100

In [ ]:
# function to train the model
def train():

    model.train()
    total_loss, total_accuracy = 0, 0

    # empty list to save model predictions
    total_preds = []

    # iterate over batches
    for step, batch in enumerate(train_dataloader):
        # Push to GPU
        batch = [r.to(device) for r in batch]
        sent_id, attention_mask, _ = batch  # We ignore original labels

        # 1. Apply Random Masking
        # sent_id is your FFT sequence
        masked_input, labels, masked_indices = apply_random_mask(sent_id)

        model.zero_grad()

        # 2. Forward pass
        preds = model(masked_input, attention_mask)

        # 3. Compute Loss ONLY on masked tokens
        # We filter the preds and labels using the boolean mask
        masked_preds = preds[masked_indices]
        masked_labels = labels[masked_indices]

        # Use MSE for continuous FFT data
        loss = F.mse_loss(masked_preds, masked_labels)

        total_loss += loss.item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # append the model predictions
    total_preds.append(preds)

    # compute the training loss of the epoch
    avg_loss = total_loss / len(train_dataloader)

    # predictions are in the form of (no. of batches, size of batch, no. of classes).
    # reshape the predictions in form of (number of samples, no. of classes)
    total_preds = torch.concatenate(total_preds, axis=0)
    # returns the loss and predictions
    return avg_loss, total_preds

In [ ]:
import time


# function for evaluating the model
def evaluate():

    print("\nEvaluating...")
    t0 = time.time()
    # deactivate dropout layers
    model.eval()

    total_loss, total_accuracy = 0, 0

    # empty list to save the model predictions
    total_preds = []

    # iterate over batches
    for step, batch in enumerate(val_dataloader):
        # Progress update every 50 batches.
        if step % 50 == 0 and not step == 0:
            # Calculate elapsed time in minutes.
            elapsed = time.time() - t0

            # Report progress.
            print("  Batch {:>5,}  of  {:>5,}.".format(step, len(val_dataloader)))

        # push the batch to gpu
        batch = [t.to(device) for t in batch]

        sent_id, attention_mask, labels = batch

        # deactivate autograd
        with torch.no_grad():
            masked_input, labels, masked_indices = apply_random_mask(sent_id)

            preds = model(masked_input, attention_mask)

            masked_preds = preds[masked_indices]
            masked_labels = labels[masked_indices]

            loss = F.mse_loss(masked_preds, masked_labels)

            total_loss += loss.item()

        total_preds.append(preds)

    # compute the validation loss of the epoch
    avg_loss = total_loss / len(val_dataloader)

    # reshape the predictions in form of (number of samples, no. of classes)
    total_preds = torch.concatenate(total_preds, axis=0)

    return avg_loss, total_preds

In [ ]:
device

device(type='mps')

In [ ]:
# set initial loss to infinite
best_valid_loss = float("inf")

# empty lists to store training and validation loss of each epoch
train_losses = []
valid_losses = []

#for each epoch
for epoch in range(epochs):
     
    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))
    
    #train model
    train_loss, _ = train()

    # evaluate model
    valid_loss, _ = evaluate()

    # save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "saved_weights.pt")

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f"\nTraining Loss: {train_loss:.3f}")
    print(f"Validation Loss: {valid_loss:.3f}")


 Epoch 1 / 100

Evaluating...

Training Loss: 722.527
Validation Loss: 616.223

 Epoch 2 / 100

Evaluating...

Training Loss: 711.198
Validation Loss: 607.162

 Epoch 3 / 100

Evaluating...

Training Loss: 704.000
Validation Loss: 598.425

 Epoch 4 / 100

Evaluating...

Training Loss: 693.031
Validation Loss: 591.403

 Epoch 5 / 100

Evaluating...

Training Loss: 682.669
Validation Loss: 584.913

 Epoch 6 / 100

Evaluating...

Training Loss: 673.346
Validation Loss: 573.252

 Epoch 7 / 100

Evaluating...

Training Loss: 662.987
Validation Loss: 568.208

 Epoch 8 / 100

Evaluating...

Training Loss: 652.975
Validation Loss: 557.162

 Epoch 9 / 100

Evaluating...

Training Loss: 644.643
Validation Loss: 550.786

 Epoch 10 / 100

Evaluating...

Training Loss: 636.695
Validation Loss: 541.037

 Epoch 11 / 100

Evaluating...

Training Loss: 629.503
Validation Loss: 539.027

 Epoch 12 / 100

Evaluating...

Training Loss: 621.703
Validation Loss: 526.147

 Epoch 13 / 100

Evaluating...

Trai

In [ ]:
# load weights of best model
path = "saved_weights.pt"
model.load_state_dict(torch.load(path))

<All keys matched successfully>

In [ ]:
from transformers import MambaConfig, MambaForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
mamba = MambaForCausalLM.from_pretrained("state-spaces/mamba-130m-hf")

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/517M [00:00<?, ?B/s]

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
class Mamba_Fourier_2(nn.Module):
    def __init__(self, backbone):
        super(Mamba_Fourier_2, self).__init__()

        self.backbone = backbone

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu = nn.ReLU()

        self.fft_projection = nn.Linear(1, 768)

        # dense layer 1
        self.fc1 = nn.Linear(768, 1)

        # dense layer 2 (Output layer)
        # self.fc2 = nn.Linear(512,2)

        # softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    # define the forward pass
    def forward(self, sent_id, mask):

        # pass the inputs to the model
        x = sent_id.to(torch.float32)
        if len(x.shape) == 2:
            x = x.unsqueeze(-1)
        elif len(x.shape) == 1:
            x = x.unsqueeze(0).unsqueeze(-1)

        x = self.fft_projection(x)

        x = self.backbone.backbone(inputs_embeds=x).last_hidden_state

        x = self.fc1(x)

        return x.squeeze(-1)

In [ ]:
model = Mamba_Fourier_2(mamba)
model.to(device)

Mamba_Fourier_2(
  (backbone): MambaForCausalLM(
    (backbone): MambaModel(
      (embeddings): Embedding(50280, 768)
      (layers): ModuleList(
        (0-23): 24 x MambaBlock(
          (norm): MambaRMSNorm(768, eps=1e-05)
          (mixer): MambaMixer(
            (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
            (act): SiLUActivation()
            (in_proj): Linear(in_features=768, out_features=3072, bias=False)
            (x_proj): Linear(in_features=1536, out_features=80, bias=False)
            (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
            (out_proj): Linear(in_features=1536, out_features=768, bias=False)
          )
        )
      )
      (norm_f): MambaRMSNorm(768, eps=1e-05)
    )
    (lm_head): Linear(in_features=768, out_features=50280, bias=False)
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (relu): ReLU()
  (fft_projection): Linear(in_features=1, out_features=768, bias=True)
  (fc1): Li

In [ ]:
# set initial loss to infinite
best_valid_loss = float("inf")

# empty lists to store training and validation loss of each epoch
train_losses = []
valid_losses = []

# for each epoch
for epoch in range(1):
    print("\n Epoch {:} / {:}".format(epoch + 1, epochs))

    # train model
    train_loss, _ = train()

    # evaluate model
    valid_loss, _ = evaluate()

    # save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "saved_weights.pt")

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f"\nTraining Loss: {train_loss:.3f}")
    print(f"Validation Loss: {valid_loss:.3f}")


 Epoch 1 / 100


KeyboardInterrupt: 